In [210]:
import pandas as pd
import numpy as np
import plotly.express as px

# Import dataset

In [211]:
DATA_PATH = "../data/raw/Walmart_Store_sales.csv"
df = pd.read_csv(DATA_PATH)

In [212]:
df.head()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,6.0,18-02-2011,1572117.54,NaN,59.61,3.045,214.777523,6.858
1,13.0,25-03-2011,1807545.43,0.0,42.38,3.435,128.616064,7.470
2,17.0,27-07-2012,NaN,0.0,NaN,NaN,130.719581,5.936
3,11.0,NaN,1244390.03,0.0,84.57,NaN,214.556497,7.346
4,6.0,28-05-2010,1644470.66,0.0,78.89,2.759,212.412888,7.092


In [213]:
df.shape

(150, 8)

In [214]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Store         150 non-null    float64
 1   Date          132 non-null    object 
 2   Weekly_Sales  136 non-null    float64
 3   Holiday_Flag  138 non-null    float64
 4   Temperature   132 non-null    float64
 5   Fuel_Price    136 non-null    float64
 6   CPI           138 non-null    float64
 7   Unemployment  135 non-null    float64
dtypes: float64(7), object(1)
memory usage: 9.5+ KB


In [215]:
df.describe(include="all")

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
count,150.000000,132,1.360000e+02,138.000000,132.000000,136.000000,138.000000,135.000000
unique,NaN,85,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,19-10-2012,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,4,NaN,NaN,NaN,NaN,NaN,NaN
mean,9.866667,NaN,1.249536e+06,0.079710,61.398106,3.320853,179.898509,7.598430
std,6.231191,NaN,6.474630e+05,0.271831,18.378901,0.478149,40.274956,1.577173
min,1.000000,NaN,2.689290e+05,0.000000,18.790000,2.514000,126.111903,5.143000
25%,4.000000,NaN,6.050757e+05,0.000000,45.587500,2.852250,131.970831,6.597500
50%,9.000000,NaN,1.261424e+06,0.000000,62.985000,3.451000,197.908893,7.470000
75%,15.750000,NaN,1.806386e+06,0.000000,76.345000,3.706250,214.934616,8.150000


# Transformations

In [216]:
# Date
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

C:\Users\jean-\AppData\Local\Temp\ipykernel_16240\3871379164.py:2: UserWarning:

Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.



In [217]:
# Split dates
df['Year'] = df['Date'].dt.year.astype("Int64")
df['Month'] = df['Date'].dt.month.astype("Int64")
df["Quarter"] = df["Date"].dt.quarter.astype("Int64")
df['Week'] = df['Date'].dt.isocalendar().week.astype("Int64")
df["Is_Year_End"] = (df["Date"].dt.month == 12).astype(int)

In [218]:
# Stores
df['Store'] = df['Store'].astype(int)

In [219]:
# Flag holiday
df['Holiday_Flag_missing'] = df['Holiday_Flag'].isna().astype(int)
df['Holiday_Flag'] = df['Holiday_Flag'].fillna(0).astype(int)

In [220]:
df.shape

(150, 14)

In [221]:
df.head()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment,Year,Month,Quarter,Week,Is_Year_End,Holiday_Flag_missing
0,6,2011-02-18,1572117.54,0,59.61,3.045,214.777523,6.858,2011,2,1,7,0,1
1,13,2011-03-25,1807545.43,0,42.38,3.435,128.616064,7.470,2011,3,1,12,0,0
2,17,2012-07-27,NaN,0,NaN,NaN,130.719581,5.936,2012,7,3,30,0,0
3,11,NaT,1244390.03,0,84.57,NaN,214.556497,7.346,<NA>,<NA>,<NA>,<NA>,0,0
4,6,2010-05-28,1644470.66,0,78.89,2.759,212.412888,7.092,2010,5,2,21,0,0


# Overview

In [222]:
# Nombre de  valeurs manquantes
valeurs_manquantes = df.isna().sum()
valeurs_manquantes

Store                    0
Date                    18
Weekly_Sales            14
Holiday_Flag             0
Temperature             18
Fuel_Price              14
CPI                     12
Unemployment            15
Year                    18
Month                   18
Quarter                 18
Week                    18
Is_Year_End              0
Holiday_Flag_missing     0
dtype: int64

In [223]:
# Pourcentage de valeurs manquantes
valeurs_manquantes_pourcent = (df.isna().sum() / df.shape[0]) * 100
valeurs_manquantes_pourcent

Store                    0.000000
Date                    12.000000
Weekly_Sales             9.333333
Holiday_Flag             0.000000
Temperature             12.000000
Fuel_Price               9.333333
CPI                      8.000000
Unemployment            10.000000
Year                    12.000000
Month                   12.000000
Quarter                 12.000000
Week                    12.000000
Is_Year_End              0.000000
Holiday_Flag_missing     0.000000
dtype: float64

In [224]:
na_pct = (df.isna().mean() * 100).sort_values(ascending=False).reset_index()
na_pct.columns = ["column", "missing_pct"]

fig = px.bar(
    na_pct,
    x="column",
    y="missing_pct",
    title="Valeurs manquantes par colonne (%)",
    labels={"column": "Colonne", "missing_pct": "% manquant"},
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [225]:
# Détection des noms de colonnes numériques ou catégorielles
numeric_features = df.select_dtypes(exclude="object").columns
categorical_features = df.select_dtypes(include="object").columns

print('Found numeric features ', numeric_features)
print('Found categorical features ', categorical_features)

Found numeric features  Index(['Store', 'Date', 'Weekly_Sales', 'Holiday_Flag', 'Temperature',
       'Fuel_Price', 'CPI', 'Unemployment', 'Year', 'Month', 'Quarter', 'Week',
       'Is_Year_End', 'Holiday_Flag_missing'],
      dtype='object')
Found categorical features  Index([], dtype='object')


In [226]:
# Vérification valeurs numériques float uniques
numeric_float = df.select_dtypes('float64')

for col in numeric_float.columns:
  print(f"Colonne {col} : ", numeric_float[col].nunique())

Colonne Weekly_Sales :  136
Colonne Temperature :  130
Colonne Fuel_Price :  120
Colonne CPI :  135
Colonne Unemployment :  104


In [227]:
# Vérification valeurs numériques int uniques
numeric_int = df.select_dtypes('int64')

for col in numeric_int.columns:
  print(f"Colonne {col} : ", numeric_int[col].nunique())

Colonne Year :  3
Colonne Month :  12
Colonne Quarter :  4
Colonne Week :  46


In [228]:
# Vérification valeurs non numériques float uniques
non_numeric = df.select_dtypes('object')

for col in non_numeric.columns:
  print(f"Colonne {col} : ", non_numeric[col].nunique())

# EDA

## Distribution de la cible Weekly_Sales

- La distribution des ventes hebdo est très étalée avec une forte hétérogénéité des niveaux de vente
- Quelques semaines à ventes très élevées probabelemnt liées à certains magasins/périodes/événements

In [229]:
fig = px.histogram(
    df.dropna(subset=["Weekly_Sales"]),
    x="Weekly_Sales",
    nbins=60,
    title="Distribution de Weekly_Sales",
    labels={"Weekly_Sales": "Weekly_Sales"},
)
fig.show()

## Détection des outliers Weekly_Sales

- La visualisation montre une dispersion importante avec une forte variabilité des ventes hebdo
- La médiane se situe autour de 1.2-1.3M avec des valeurs typiques qui s’étendent approximativement de  0.6M à 1.8M

In [230]:
fig = px.box(
    df.dropna(subset=["Weekly_Sales"]),
    y="Weekly_Sales",
    title="Outliers sur Weekly_Sales (boxplot)",
    labels={"Weekly_Sales": "Weekly_Sales"},
)
fig.show()

## Série temporelle (Weekly_Sales agrégé par Date)

- La série montre une forte variabilité avec des pics récurrents suggérant un effet saisonnier/événementiel (périodes de fêtes, promotions)
- Il existe  des pointes marquées autour de la fin d’année à relier avec un impact potentiel de la variable de Holiday_Flag et/ou de la saison (Month, WeekOfYear)

In [231]:
ts = (
    df.dropna(subset=["Date", "Weekly_Sales"])
      .groupby("Date", as_index=False)["Weekly_Sales"].sum()
      .sort_values("Date")
)

fig = px.line(
    ts,
    x="Date",
    y="Weekly_Sales",
    title="Weekly_Sales total (somme) au fil du temps",
    labels={"Weekly_Sales": "Ventes hebdo (somme)"},
)
fig.show()

## Saisonnalité par mois

- Le boxplot par mois confirme une saisonnalité nette : les niveaux de ventes et leur dispersion varient fortement selon le mois
- Le mois de décembre (12) ressort nettement avec les médianes les plus élevées et une variabilité importante cohérente avec un effet “fin d’année / fêtes”

In [232]:
fig = px.box(
    df.dropna(subset=["Month", "Weekly_Sales"]),
    x="Month",
    y="Weekly_Sales",
    title="Saisonnalité : Weekly_Sales par mois",
    labels={"Month": "Mois", "Weekly_Sales": "Weekly_Sales"},
)
fig.show()

## Impact Holiday vs Non-holiday

- Les semaines flaguées Holiday (1) présentent une médiane plus élevée que les semaines non-holiday ce qui semble confirmer un effet positif des périodes de fêtes sur les ventes
- La distribution semble aussi plus concentrée (moins de dispersion apparente) alors que le non-holiday est plus hétérogène

In [233]:
fig = px.box(
    df.dropna(subset=["Holiday_Flag", "Weekly_Sales"]),
    x="Holiday_Flag",
    y="Weekly_Sales",
    points="outliers",
    title="Impact Holiday_Flag sur Weekly_Sales",
    labels={"Holiday_Flag": "Holiday_Flag (0/1)", "Weekly_Sales": "Weekly_Sales"},
)
fig.show()

## Top 10 magasins par ventes moyennes

- Le top 10 met en évidence une forte hétérogénéité inter-magasins : certains magasins ont des ventes moyennes nettement supérieures (1.4M à 2.1M)

In [234]:
store_stats = (
    df.dropna(subset=["Store", "Weekly_Sales"])
      .groupby("Store", as_index=False)["Weekly_Sales"].mean()
      .sort_values("Weekly_Sales", ascending=False)
      .head(10)
)

# IMPORTANT : Store en catégorie pour éviter l’axe continu
store_stats["Store"] = store_stats["Store"].astype(str)

fig = px.bar(
    store_stats,
    x="Store",
    y="Weekly_Sales",
    title="Top 10 Stores par Weekly_Sales moyenne",
    labels={"Weekly_Sales": "Weekly_Sales moyenne"},
    category_orders={"Store": store_stats["Store"].tolist()}  # respecte l'ordre du top
)

fig.show()

## Relation cible vs variable numérique

### Temperature

- La visualisation ne montre pas de relation linéaire claire entre Temperature et Weekly_Sales : la dispersion est élevée à toutes les températures
- S’il existe un effet, il est probablement faible, non linéaire ou masqué par des facteurs plus dominants (Store, saisonnamlité, holidays)

In [235]:
fig = px.scatter(
    df.dropna(subset=["Temperature", "Weekly_Sales"]),
    x="Temperature",
    y="Weekly_Sales",
    opacity=0.35,
    title="Weekly_Sales vs Temperature",
    labels={"Temperature": "Temperature", "Weekly_Sales": "Weekly_Sales"},
)
fig.show()

### Fuel_Price

- La visualisation ne met pas en évidence de tendance linéaire nette entre Fuel_Price et Weekly_Sales : la variabilité des ventes reste élevée quel que soit le prix du carburant

In [236]:
# Fuel_Price
fig = px.scatter(
    df.dropna(subset=["Temperature", "Weekly_Sales"]),
    x="Fuel_Price",
    y="Weekly_Sales",
    opacity=0.35,
    title="Weekly_Sales vs Fuel_Price",
    labels={"Fuel_Price": "Fuel_Price", "Weekly_Sales": "Weekly_Sales"},
)
fig.show()

### CPI

- On observe des clusters très marqués de valeurs de CPI ce qui pourrait suggèrer que CPI varie surtout par période (et/ou par store)
- Pas de relation évidente entre CPI et Weekly_Sales : pour un même niveau de CPI, les ventes restent très dispersées

In [237]:
fig = px.scatter(
    df.dropna(subset=["Temperature", "Weekly_Sales"]),
    x="CPI",
    y="Weekly_Sales",
    opacity=0.35,
    title="Weekly_Sales vs CPI",
    labels={"CPI": "CPI", "Weekly_Sales": "Weekly_Sales"},
)
fig.show()

### Unemployment

- La visualisation ne montre pas de relation linéaire évidente : les ventes restent très dispersées pour des niveaux de chômage similaires
- On distingue une concentration principale autour de 6 à 9% de chômage, avec quelques valeurs plus élevées (13 à 15%) peu représentées

In [238]:
# Unemployment
fig = px.scatter(
    df.dropna(subset=["Temperature", "Weekly_Sales"]),
    x="Unemployment",
    y="Weekly_Sales",
    opacity=0.35,
    title="Weekly_Sales vs Unemployment",
    labels={"Unemployment": "Unemployment", "Weekly_Sales": "Weekly_Sales"},
)
fig.show()

## Heatmap de corrélations

- On observe une forte colinéarité Fuel_Price / Year (0.81) et une corrélation CPI / Unemployment (≈ -0.35)

In [239]:
num_cols = [c for c in ["Weekly_Sales","Temperature","Fuel_Price","CPI","Unemployment","Year","Month","Day","DayOfWeek"] if c in df.columns]
corr = df[num_cols].corr(numeric_only=True).round(2)

fig = px.imshow(
    corr,
    text_auto=True,
    aspect="auto",
    zmin=-1, zmax=1,   # échelle cohérente
    title="Corrélations (numériques)"
)

fig.update_layout(
    width=950, height=650,
    margin=dict(l=80, r=40, t=80, b=80)
)
fig.update_xaxes(tickangle=-45)
fig.update_traces(textfont_size=12)

fig.show()

## Export du dataset pour Preprocessing et feature Engineering

In [240]:
out = df.copy()
out.to_csv("../data/outputs/Walmart_Store_sales_ml_output.csv", index=False)